# 01. Trend Analysis: Understanding Patterns Over Time
### Primary Analytical Purpose: *How does the metric change over time?*
**Lecture Reference:** Purpose 1 — Trend (Slides 8–12, 45)

Trend analysis investigates how continuous quantitative metrics evolve over an ordered temporal dimension (days, weeks, months, quarters, years). Rather than comparing isolated numbers, trend analysis reveals:
- **Direction & Slope:** Is performance increasing, stagnating, or declining? Steeper slope signifies faster rate of change.
- **Seasonality & Cycles:** Are there recurring patterns, such as winter peaks or summer lulls?
- **Inflection Points & Turning Points:** When did the growth rate change or shift direction?

---

## 1. Scenario and Dataset Preview
**Business Scenario:** Qatar Tourism Authority monitors monthly international visitor inflows (in thousands) and hotel occupancy rates across 2024. The objective is to evaluate seasonal fluctuations to plan hotel capacity, public transportation schedules, and national festival programming.

The dataset is loaded from `data/qatar_monthly_visitors.csv`. Let us inspect the data structure and first few records.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Set modern Seaborn theme
sns.set_theme(style="whitegrid")

# Load dataset from data folder
df = pd.read_csv("data/qatar_monthly_visitors.csv")

# Display the dataset
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df

## 2. Primary Visualization: Line Chart (The Default)
- **What the chart shows:** Position along a continuous time scale connected by line segments, showing slope and trajectory.
- **When it is useful:** For 5 or more continuous time points; the gold standard for tracking direction, rate of change, and seasonality.
- **Why chosen:** Position along a continuous scale is the most accurately decoded geometric visual attribute for time series.

Below, we demonstrate both **Seaborn** (static, report-ready) and **Plotly Express** (interactive, dashboard-ready). Notice that in accordance with Principle 2 and Slide 5, the chart title states the **finding**, not merely the axes.

In [ ]:
# --- Seaborn Implementation: Static Publication Chart ---
plt.figure(figsize=(9, 4.5))

# Plot monthly visitor line with markers
ax = sns.lineplot(
    data=df, 
    x="month", 
    y="visitors_thousands", 
    marker="o", 
    linewidth=2.5, 
    color="#14A098"
)

# Principle 4 (Honest Axes): Start value axis at zero for volume measurements
ax.set_ylim(0, 500)

# Principle 2 & 10: Clear titles stating the finding; clear axis labels
ax.set_title("Qatar International Visitors Peaked in Winter (Nov-Dec 2024)", fontsize=13, weight="bold")
ax.set_xlabel("Month (2024)", fontsize=11)
ax.set_ylabel("Visitors (Thousands)", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# --- Plotly Express Implementation: Interactive Dashboard Chart ---
fig = px.line(
    df, 
    x="month", 
    y="visitors_thousands", 
    markers=True,
    title="Qatar International Visitors Peaked in Winter (Nov-Dec 2024)",
    labels={"month": "Month (2024)", "visitors_thousands": "Visitors (Thousands)"}
)

# Set honest axis range starting at zero
fig.update_yaxes(range=[0, 500])
fig.update_traces(line_color="#14A098", line_width=3, marker_size=8)
fig.show()

### Structured Interpretation: The 5-Step Framework
Following the lecture's five-step insight structure (Slides 42–44):

1. **Question:** When does Qatar experience peak international visitor arrivals, and how severe is the summer slowdown?
2. **Visualization:** Monthly line chart with markers, plotted over a continuous zero-based scale.
3. **Observation:** Visitors peaked at 460k in December and 420k in November, while falling to a low of 140k in July.
4. **Insight:** Visitor inflows exhibit pronounced seasonality driven by the winter weather window, with summer arrivals dropping ~70% below December levels.
5. **Implication:** Hospitality and transport operators should schedule heavy maintenance in June–July and scale staff and services for November–February.

## 3. Alternative 1: Area Chart (Cumulative Magnitude)
- **What the chart shows:** The shaded area beneath the line emphasizes the cumulative volume of visitors over the entire period.
- **When to use instead:** Only when the magnitude of a single total matters, or for stacked totals.
- **Rule of thumb (Slide 9):** Avoid overlapping areas for multiple series, as background layers become obscured.

In [ ]:
# --- Plotly Express: Area Chart ---
fig_area = px.area(
    df, 
    x="month", 
    y="visitors_thousands",
    title="Cumulative Volume: Over 3.2M Visitors Traveled to Qatar Across 2024",
    labels={"month": "Month (2024)", "visitors_thousands": "Visitors (Thousands)"}
)

fig_area.update_traces(line_color="#14A098", fillcolor="rgba(20, 160, 152, 0.35)")
fig_area.update_yaxes(range=[0, 500])
fig_area.show()

### Interpretation for Area Chart
- **Observation:** The filled area widens significantly during Q1 (Jan–Mar) and Q4 (Oct–Dec), while dipping into a narrow trough in mid-summer (Jul–Aug).
- **Derived Insight:** Total annual visitor volume reached ~3.27 million, but over 69% of that total arrived during the cooler six months.

## 4. Alternative 2: Column Chart (Few Discrete Periods)
- **What the chart shows:** Total quarterly arrivals displayed as distinct vertical bars.
- **When to use instead:** When time points are few and discrete (e.g., 4 quarters) and each period represents a separate reporting event rather than a continuous daily flow (Slide 9 & 45).

In [ ]:
# Aggregate monthly data into quarterly totals
quarterly_df = df.groupby("quarter", as_index=False)["visitors_thousands"].sum()

# --- Seaborn: Quarterly Column Chart ---
plt.figure(figsize=(7, 4.2))
ax_col = sns.barplot(
    data=quarterly_df, 
    x="quarter", 
    y="visitors_thousands", 
    color="#14A098"  # Principle 7: Single intentional colour
)

# Principle 4: Honest zero baseline
ax_col.set_ylim(0, 1400)
ax_col.set_title("Quarterly Totals: Q1 and Q4 Accounted for 69% of Total 2024 Visitors", fontsize=12, weight="bold")
ax_col.set_xlabel("Quarter", fontsize=11)
ax_col.set_ylabel("Total Visitors (Thousands)", fontsize=11)

# Annotate each bar directly with exact values
for p in ax_col.patches:
    height = p.get_height()
    ax_col.annotate(
        f"{int(height):,}k", 
        (p.get_x() + p.get_width() / 2., height + 25), 
        ha='center', 
        fontsize=10, 
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

### Interpretation for Column Chart
- **Observation:** Q4 led with 1,190k visitors, followed closely by Q1 at 1,137k. Q3 had the lowest quarterly volume at 515k.
- **Derived Insight:** Evaluating discrete quarters confirms that the winter half of the year generates more than double the volume of the summer half.

## 5. Best Practices & Common Mistakes (Lecture Principles)
**Visualization Best Practices (Slide 11 & 46):**
- **Keep time on the x-axis**, running from oldest on the left to newest on the right.
- **Start the value axis at zero** for amounts and volumes; relax only for standardized indices and state why.
- **State the aggregation grain in the title:** daily, weekly, monthly, or quarterly.
- **Limit multi-line plots to 3–5 lines;** highlight the key line and mute the rest as context.

**Common Problems & Misleading Designs to Avoid:**
- **Truncated y-axis:** Starting the y-axis at 120k would exaggerate a normal seasonal dip into an alarming collapse.
- **Uneven intervals plotted evenly:** Spacing irregular dates evenly distorts the perceived slope.
- **Extrapolating beyond observed data:** A 12-month series cannot reliably forecast years ahead without multi-year cycle validation.

## 6. Hands-on Practice Exercises

### Exercise 1: Chart Interpretation
Inspect the `hotel_occupancy_pct` column in `df`. In your own words:
1. **Observation:** What were the highest and lowest occupancy months?
2. **Insight:** How closely does hotel occupancy track international visitor arrivals?
3. **Implication:** What seasonal pricing strategy should Qatar hotels adopt?

### Exercise 2: Code Modification
Plot a line chart of `hotel_occupancy_pct` across the 12 months using Seaborn:
- Set the y-axis limits between 0% and 100%.
- Add markers and a title that states the analytical finding.

In [ ]:
# Exercise 2: Complete the code below to plot hotel occupancy percentage
plt.figure(figsize=(9, 4.5))

# Plot 'hotel_occupancy_pct' vs 'month'
# ax = sns.lineplot(data=df, x='month', y='hotel_occupancy_pct', marker='s', color='#2B5C8F')
# ax.set_ylim(0, 100)
# ax.set_title(...)

plt.show()

### Exercise 3: Discrete Aggregation
Using `df.groupby('quarter')['hotel_occupancy_pct'].mean()`, calculate the average hotel occupancy by quarter and create an interactive bar chart using Plotly Express `px.bar()`.

In [ ]:
# Exercise 3: Complete the quarterly occupancy calculation and bar plot
# quarterly_occ = df.groupby('quarter', as_index=False)['hotel_occupancy_pct'].mean()
# fig = px.bar(quarterly_occ, x='quarter', y='hotel_occupancy_pct', text_auto='.1f', title=...)
# fig.show()

## 7. Key Takeaways
1. **Line charts are the default for trend analysis** because position along continuous time accurately communicates slope and seasonality.
2. **Area charts emphasize cumulative magnitude**, while column charts are best when comparing a few discrete reporting periods.
3. **Always anchor the value axis at zero** for amounts and counts to preserve honest visual proportions.
4. **The title is where the analysis lands:** State the finding in the title, not just the axes.